# Chạy QG-RSIAT trên Kaggle

Chọn **GPU** trong Notebook options và thêm Kaggle Dataset chứa dữ liệu vào **Notebook > Input**. Checkpoint ViT-IN21K nằm sẵn trong thư mục `vit-base-patch16-224-in21k` của repository; không cần thêm checkpoint riêng vào Kaggle Input. Vì checkpoint được lưu bằng Git LFS, cell clone repository sẽ tải file này một lần từ GitHub LFS. Khi train, chương trình đọc file local đã clone, không tải weights qua `timm`.

CIFAR-100 cần thư mục `cifar-100-python` gồm `train`, `test`, `meta`; ImageNet-A/R cần `train/<class>` và `test/<class>`. Chạy các cell theo thứ tự từ trên xuống. Log được lưu trong `/kaggle/working/QG_RSIAT/logs/`. Nếu thay đổi code trong VS Code, cập nhật repository/notebook Kaggle trước khi chạy để notebook clone đúng phiên bản.

In [ ]:
!git clone --depth 1 https://github.com/deety03503/QG_RSIAT.git /kaggle/working/QG_RSIAT

In [ ]:
!python -m pip install --quiet "timm==0.6.12" "easydict==1.13"
!nvidia-smi
!python -m pip show torch torchvision timm easydict

## Nối dataset

Chọn `DATASET`, rồi đặt `DATASET_DIR` thành đường dẫn chính xác tới thư mục dữ liệu trong `/kaggle/input`. Với CIFAR-100, trỏ thẳng tới thư mục Python đã giải nén `cifar-100-python`, chứa các file `train`, `test`, `meta`; loader đọc trực tiếp từ đó và không tải/copy dữ liệu. Với ImageNet-A/R, dùng thư mục có `train/<class>` và `test/<class>`. ViT-IN21K sẽ được nạp từ checkpoint đã clone trong `vit-base-patch16-224-in21k`; không cần cấu hình đường dẫn trọng số riêng.

In [ ]:
# Chọn dataset và thay đường dẫn mẫu bằng đường dẫn trong /kaggle/input.
DATASET = "cifar224"  # cifar224, imageneta hoặc imagenetr
RUN_MODE = "quick"  # chỉ áp dụng cho cifar224: quick hoặc full
DATASET_DIR = "/kaggle/input/your-dataset/cifar-100-python"

DATASET_NAME = {"cifar224": "cifar-100-python", "imageneta": "imagenet-a", "imagenetr": "imagenet-r"}[DATASET]
CONFIG_NAME = f"adapter_{DATASET}"

from pathlib import Path

dataset_source = Path(DATASET_DIR).expanduser()
if not dataset_source.is_dir():
    raise FileNotFoundError(f"Dataset directory not found: {dataset_source}")
if DATASET == "cifar224":
    missing_files = [name for name in ("train", "test", "meta") if not (dataset_source / name).is_file()]
    if missing_files:
        raise FileNotFoundError(f"CIFAR-100 folder is missing required files: {missing_files}")
checkpoint_source = Path("/kaggle/working/QG_RSIAT/vit-base-patch16-224-in21k/pytorch_model.bin")
if not checkpoint_source.is_file():
    raise FileNotFoundError(f"Repository ViT-IN21K checkpoint not found: {checkpoint_source}. Check that Git LFS completed during clone.")

if DATASET == "cifar224":
    import os
    if dataset_source.name != "cifar-100-python":
        raise ValueError(f"DATASET_DIR must point to the extracted cifar-100-python directory, got: {dataset_source}")
    os.environ["RSIAT_CIFAR100_DIR"] = str(dataset_source.resolve())
    print(f"Using CIFAR-100 directly from Kaggle Input: {dataset_source}")
else:
    dataset_target = Path(f"/kaggle/working/QG_RSIAT/data/datasets/{DATASET_NAME}")
    dataset_target.parent.mkdir(parents=True, exist_ok=True)
    if dataset_target.is_symlink():
        dataset_target.unlink()
    elif dataset_target.exists():
        raise FileExistsError(f"Dataset target exists and is not a symlink: {dataset_target}")
    dataset_target.symlink_to(dataset_source.resolve(), target_is_directory=True)

if DATASET != "cifar224":
    print(f"Connected {dataset_source} to {dataset_target}")
print(f"Using repository-local ViT checkpoint: {checkpoint_source}")

## CONFIG OVERRIDE

Cell tiếp theo sao chép `adapter_{DATASET}.json` thành config Kaggle riêng và áp dụng thiết lập GPU, số worker, seed. Với `cifar224`, `RUN_MODE="quick"` chạy 3 epoch khởi tạo + 3 epoch cho mỗi task và bỏ bước compact-classifier để kiểm tra nhanh; kết quả quick không dùng để so sánh benchmark. Dùng `RUN_MODE="full"` để giữ nguyên cấu hình benchmark. Config gốc không bị thay đổi.

In [ ]:
import json
from pathlib import Path

base_config = Path(f"/kaggle/working/QG_RSIAT/exps/{CONFIG_NAME}.json")
config = json.loads(base_config.read_text())
config.update({"device": ["0"], "num_worker": 2, "seed": [1993]})
if RUN_MODE not in {"quick", "full"}:
    raise ValueError("RUN_MODE must be 'quick' or 'full'")
if DATASET == "cifar224" and RUN_MODE == "quick":
    config.update({"init_epochs": 3, "warmup_epoch": 1, "inc_epochs": 3, "ca_epochs": 0, "ca": False})
output_config = base_config.with_name(f"{CONFIG_NAME}.kaggle.json")
output_config.write_text(json.dumps(config, indent=2) + "\n")
print(f"Config ready ({RUN_MODE}): {output_config}")

In [ ]:
%cd /kaggle/working/QG_RSIAT
!python main.py --config "exps/{CONFIG_NAME}.kaggle.json"